In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

RAW = "../data/raw/"

# --- leitura e limpeza (mesmo padrão do merge_datasets.py) ---
ncm_raw = pd.read_csv(
    RAW + "comexstat_ncm_santos_2021-09_10.csv",
    sep=";",
    encoding="utf-8-sig",
)
for coluna in ncm_raw.columns:
    # checa "não numérico" em vez de "dtype == object": no pandas >= 2.x com a
    # opção de string dtype (e default no pandas 3.x), colunas de texto não são
    # mais "object" — usar só "== object" faz esse loop silenciosamente não
    # limpar nada e deixar sujeira de \r escondida no meio do texto
    if not pd.api.types.is_numeric_dtype(ncm_raw[coluna]):
        ncm_raw[coluna] = ncm_raw[coluna].astype(str).str.strip().str.replace("\r", "", regex=False)
ncm_raw.columns = [c.strip() for c in ncm_raw.columns]

ncm_raw["ano"] = ncm_raw["Ano"].astype(int)
ncm_raw["mes"] = ncm_raw["M\u00eas"].str.extract(r"^(\d+)").astype(int)
ncm_raw["ncm"] = ncm_raw["C\u00f3digo NCM"].astype(str).str.zfill(8)
ncm_raw["capitulo"] = ncm_raw["ncm"].str[:2]
ncm_raw["descricao"] = ncm_raw["Descri\u00e7\u00e3o NCM"]
ncm_raw["fob"] = ncm_raw["Valor US$ FOB"].astype(float)
ncm_raw["kg"] = ncm_raw["Quilograma L\u00edquido"].astype(float)

meses_nome = {9: "set/2021", 10: "out/2021"}
ncm_raw["mes_nome"] = ncm_raw["mes"].map(meses_nome)

ncm_raw[["ano", "mes_nome", "ncm", "descricao", "fob", "kg", "capitulo"]].head()


In [ ]:
display(Markdown(
    "## Detalhamento por produto \u2014 set/out de 2021\n\n"
    "No notebook `01_eda_painel_2015_2025`, o boxplot de `importa\u00e7\u00e3o_kg` apontou "
    "set/2021 e out/2021 como os \u00fanicos meses at\u00edpicos entre as vari\u00e1veis de "
    "com\u00e9rcio exterior \u2014 na verdade, os dois maiores volumes de importa\u00e7\u00e3o (em kg) "
    "de toda a s\u00e9rie 2015\u20132025. O c\u00e2mbio estava historicamente alto nesses meses "
    "(acima do Q3), o que a princ\u00edpio pareceria desestimular importa\u00e7\u00e3o, mas o "
    "volume f\u00edsico foi recorde. A hip\u00f3tese levantada, a partir do pre\u00e7o m\u00e9dio "
    "impl\u00edcito por kg (que caiu para os menores n\u00edveis da s\u00e9rie nesses dois meses), "
    "era de que a carga que entrou era de baixo valor por quilo \u2014 t\u00edpico de "
    "commodity/insumo agr\u00edcola, n\u00e3o de bem manufaturado.\n\n"
    "Esse notebook usa o detalhamento por NCM (Comex Stat) desses dois meses "
    "espec\u00edficos pra checar essa hip\u00f3tese com dado real de produto, em vez de "
    "infer\u00eancia indireta."
))


In [ ]:
validacao = ncm_raw.groupby("mes_nome")[["fob", "kg"]].sum()

display(Markdown("### Confer\u00eancia: os totais batem com o painel?"))
display(validacao.style.format("{:,.0f}"))
display(Markdown(
    "Os totais de `fob` e `kg` agregados aqui batem exatamente com "
    "`importa\u00e7\u00e3o_fob` e `importa\u00e7\u00e3o_kg` do painel pra set e out/2021 \u2014 "
    "confirma que esse recorte de NCM \u00e9 o mesmo universo de carga, s\u00f3 "
    "aberto por produto."
))


In [ ]:
top_n = 15
top_ncm = (
    ncm_raw.groupby(["ncm", "descricao"])["kg"]
    .sum()
    .sort_values(ascending=False)
    .head(top_n)
    .reset_index()
)
top_ncm["kg_mi_t"] = top_ncm["kg"] / 1000 / 1e6  # peso em milh\u00f5es de toneladas

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top_ncm["descricao"].str.slice(0, 45)[::-1], top_ncm["kg_mi_t"][::-1], color="steelblue")
ax.set_xlabel("Milh\u00f5es de toneladas (set+out/2021)")
ax.set_title(f"Top {top_n} produtos (NCM) por peso \u2014 importa\u00e7\u00e3o, set+out/2021")
plt.tight_layout()
plt.show()

pct_top = top_ncm["kg"].sum() / ncm_raw["kg"].sum() * 100
display(Markdown(
    f"Esses {top_n} produtos sozinhos respondem por **{pct_top:.1f}%** do peso total "
    f"importado nos dois meses (de {ncm_raw['ncm'].nunique()} NCMs diferentes que "
    "aparecem no recorte)."
))


In [ ]:
nomes_capitulo = {
    "31": "Adubos e fertilizantes",
    "25": "Sal, enxofre, terras/pedras, gesso, cal, cimento",
    "27": "Combust\u00edveis minerais, \u00f3leos minerais",
    "28": "Produtos qu\u00edmicos inorg\u00e2nicos",
    "29": "Produtos qu\u00edmicos org\u00e2nicos",
    "10": "Cereais",
    "39": "Pl\u00e1sticos e suas obras",
    "84": "M\u00e1quinas e equipamentos mec\u00e2nicos",
    "72": "Ferro e a\u00e7o",
    "38": "Produtos diversos das ind\u00fastrias qu\u00edmicas",
}

por_capitulo = ncm_raw.groupby("capitulo")["kg"].sum().sort_values(ascending=False)
top_capitulos = por_capitulo.head(10)
pct_capitulos = top_capitulos / ncm_raw["kg"].sum() * 100

fig, ax = plt.subplots(figsize=(8, 5))
labels = [f"{c} \u2014 {nomes_capitulo.get(c, 'outro')}" for c in pct_capitulos.index]
ax.barh(labels[::-1], pct_capitulos.values[::-1], color="darkorange")
ax.set_xlabel("% do peso total importado (set+out/2021)")
ax.set_title("Participa\u00e7\u00e3o por cap\u00edtulo do SH \u2014 importa\u00e7\u00e3o, set+out/2021")
plt.tight_layout()
plt.show()

fertilizante_pct = por_capitulo.reindex(["31", "25", "28"]).sum() / ncm_raw["kg"].sum() * 100
display(Markdown(
    f"Sozinho, o cap\u00edtulo **31 (adubos e fertilizantes)** responde por "
    f"**{por_capitulo['31'] / ncm_raw['kg'].sum() * 100:.1f}%** do peso importado nos dois meses. "
    "Somando os cap\u00edtulos ligados \u00e0 cadeia de fertilizante/insumo mineral "
    "(31 \u2014 adubos, 25 \u2014 enxofre/sal, 28 \u2014 qu\u00edmicos inorg\u00e2nicos como soda c\u00e1ustica), "
    f"chega a **{fertilizante_pct:.1f}%** do volume total.\n\n"
    "Isso confirma a hip\u00f3tese: o recorde de importa\u00e7\u00e3o em kg de set/out de 2021 "
    "n\u00e3o veio de bens manufaturados sens\u00edveis a pre\u00e7o/c\u00e2mbio, e sim de uma "
    "concentra\u00e7\u00e3o muito forte em fertilizante e seus insumos \u2014 carga pesada e "
    "de baixo valor por quilo, comprada porque a safra de ver\u00e3o 2021/22 "
    "(soja/milho) precisava do insumo no ch\u00e3o antes do plantio, "
    "independentemente do c\u00e2mbio estar caro."
))
